1️⃣ What Is Reflection Pattern?
The Reflection Pattern is when an AI agent:
- Generates an initial output
- Critiques its own output
- Improves it
- Repeats until quality threshold is met

It’s basically:
```
Think → Review → Improve → Repeat
```
Instead of:
```
Think → Output
```
2️⃣ Why It Matters (Especially for You)
Reflection pattern helps with:
- Reducing hallucinations
- Improving reasoning quality
- Better SQL generation
- Cleaner code generation
- Better structured outputs
- Self-debugging


| Benefit                 | Cost            |
| ----------------------- | --------------- |
| Better accuracy         | More tokens     |
| Fewer hallucinations    | Higher latency  |
| More structured outputs | More complexity |

For production systems → use smaller cheap model for critique.

In [40]:
from dotenv import load_dotenv
load_dotenv()


from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",   # fast + cheap
    temperature=0.2
)

In [42]:
from langchain_core.prompts import ChatPromptTemplate

generator_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert software engineer."),
    ("human", "Solve the following problem:\n{question}")
])

critic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict code reviewer."),
    ("human", """
Review the following answer:

{answer}

Check:
- Logical correctness
- Edge cases
- Missing explanations

If correct, respond only with: APPROVED
Otherwise explain what is wrong.
""")
])

improver_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert engineer improving previous work."),
    ("human", """
Improve the answer based on this feedback:

Feedback:
{feedback}

Original Answer:
{answer}
""")
])

In [43]:
from typing import TypedDict

class GraphState(TypedDict):
    question: str
    answer: str
    feedback: str
    approved: bool

In [44]:
def generate(state: GraphState):
    chain = generator_prompt | llm
    response = chain.invoke({"question": state["question"]})
    
    return {
        "answer": response.content,
        "approved": False
    }

def critique(state: GraphState):
    chain = critic_prompt | llm
    response = chain.invoke({"answer": state["answer"]})
    
    content = response.content
    
    if "APPROVED" in content:
        return {
            "approved": True,
            "feedback": content
        }
    
    return {
        "approved": False,
        "feedback": content
    }

def improve(state: GraphState):
    chain = improver_prompt | llm
    response = chain.invoke({
        "answer": state["answer"],
        "feedback": state["feedback"]
    })
    
    return {
        "answer": response.content
    }



In [45]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(GraphState)

workflow.add_node("generate", generate)
workflow.add_node("critique", critique)
workflow.add_node("improve", improve)

workflow.set_entry_point("generate")

workflow.add_edge("generate", "critique")

# Conditional edge
def should_continue(state: GraphState):
    if state["approved"]:
        return END
    return "improve"

workflow.add_conditional_edges(
    "critique",
    should_continue
)

workflow.add_edge("improve", "critique")

graph = workflow.compile()

In [46]:
result = graph.invoke({
    "question": "Write a Python function to check if a number is prime."
})

print(result["answer"])

Certainly! A prime number is a natural number greater than 1 that cannot be formed by multiplying two smaller natural numbers. In other words, a prime number has exactly two distinct positive divisors: 1 and itself.

Here’s a Python function to check if a number is prime:

```python
def is_prime(n):
    """Check if a number is prime."""
    if n <= 1:
        return False  # 0 and 1 are not prime numbers
    if n <= 3:
        return True   # 2 and 3 are prime numbers
    if n % 2 == 0 or n % 3 == 0:
        return False  # Eliminate multiples of 2 and 3

    # Check for factors from 5 to the square root of n
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6  # Check only numbers of the form 6k ± 1

    return True

# Example usage:
number = 29
if is_prime(number):
    print(f"{number} is a prime number.")
else:
    print(f"{number} is not a prime number.")
```

### Explanation:
1. **Initial Checks**:
   - If `n` is less 